# Stockholm Bus Deadhead Analysis

This notebook fetches GTFS-RT vehicle position data from the Trafiklab KoDa API and analyzes deadhead (tomkörning) patterns for Stockholm bus routes.

**Works in:** Google Colab, GitHub Codespaces, or any local Jupyter environment.

## 1. Environment Setup

Installs dependencies and sets up the project path. Handles both Colab and Codespaces automatically.

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repo into Colab (or pull latest if already cloned)
    # Set GITHUB_TOKEN to enable push: os.environ["GITHUB_TOKEN"] = "ghp_..."
    REPO_BRANCH = "claude/stockholm-bus-analysis-qxUV9"
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
    if GITHUB_TOKEN:
        REPO_URL = f"https://{GITHUB_TOKEN}@github.com/HEVI-SE/Trafiklab.git"
    else:
        REPO_URL = "https://github.com/HEVI-SE/Trafiklab.git"
    REPO_DIR = "/content/Trafiklab"
    if not os.path.exists(REPO_DIR):
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull origin {REPO_BRANCH}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    # Re-set remote URL with token so push works
    if GITHUB_TOKEN:
        !cd {REPO_DIR} && git remote set-url origin {REPO_URL}
    print(f"Colab: working directory set to {REPO_DIR}")
    if not GITHUB_TOKEN:
        print("⚠ GITHUB_TOKEN ej satt — git push kommer inte fungera. Sätt os.environ['GITHUB_TOKEN'] = 'ghp_...'")
else:
    # Codespaces / local: ensure we're in the repo root
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    if os.path.exists(os.path.join(notebook_dir, "config.py")):
        os.chdir(notebook_dir)
        sys.path.insert(0, notebook_dir)
    print(f"Local: working directory is {os.getcwd()}")

# Install dependencies
!pip install -q pandas requests py7zr plotly gtfs-realtime-bindings

print("Setup complete!")

## 2. Configuration

Set the date and hours you want to analyze. The API key has a default but you can override it.

In [ ]:
# ---- EDIT THESE ----
START_DATE = "2025-03-17"     # Start date (YYYY-MM-DD)
END_DATE = "2025-03-17"       # End date (same as start for single day)
HOURS = list(range(6, 22))    # Hours to fetch per day (6 AM to 9 PM)

# Optional: override the default API key
# os.environ["KODA_API_KEY"] = "your_key_here"

# Generate date range
from datetime import datetime, timedelta
_start = datetime.strptime(START_DATE, "%Y-%m-%d")
_end = datetime.strptime(END_DATE, "%Y-%m-%d")
DATES = []
while _start <= _end:
    DATES.append(_start.strftime("%Y-%m-%d"))
    _start += timedelta(days=1)

print(f"Will analyze: {len(DATES)} day(s) ({DATES[0]} to {DATES[-1]}), hours {HOURS[0]:02d}-{HOURS[-1]:02d}")

## 3. Load Static GTFS Schedule

Downloads the static GTFS data (routes, trips, stops, stop_times) for the selected date.

In [ ]:
import pandas as pd
from config import OPERATOR_MAPPING
from fetcher import load_static_gtfs, build_trip_lookup

# Use GTFS from first date (valid for the whole range)
routes, trips, stops, stop_times = load_static_gtfs(DATES[0])

print(f"Routes: {len(routes)}, Trips: {len(trips)}, Stops: {len(stops)}, Stop times: {len(stop_times)}")

## 4. Build Trip Lookup Table

Creates a lookup mapping trip_id to route, operator, headsign, first/last stop.

In [ ]:
operator_df = pd.DataFrame(OPERATOR_MAPPING)
trip_lookup = build_trip_lookup(trips, routes, operator_df, stop_times, stops)

print(f"Trip lookup: {len(trip_lookup)} trips")
print(f"Operators: {trip_lookup['operator'].value_counts().to_dict()}")
trip_lookup.head()

## 5. Fetch Vehicle Positions

Downloads GTFS-RT vehicle positions hour by hour and builds movement segments.

**Note:** This step can take 10-30 minutes depending on the number of hours. The KoDa API may return 202 (generating archive) and the code will retry automatically.

In [ ]:
from fetcher import fetch_vehicle_positions, filter_bus_segments
from csv_handler import load_segments, get_already_fetched_date_hours

# Load any previously cached segments from CSV (committed to repo)
cached_segments = load_segments()
already_fetched = get_already_fetched_date_hours()

if already_fetched:
    total_requested = sum(1 for d in DATES for h in HOURS)
    already_count = sum(1 for d in DATES for h in HOURS if (d, h) in already_fetched)
    print(f"Redan cachade: {already_count}/{total_requested} datum+timmar")

# Only fetch what's missing
all_segments = []
for date in DATES:
    missing_hours = [h for h in HOURS if (date, h) not in already_fetched]
    if not missing_hours:
        print(f"\n=== {date} === (alla timmar cachade, hoppar över)")
        continue
    print(f"\n=== {date} === (hämtar {len(missing_hours)} av {len(HOURS)} timmar)")
    seg = fetch_vehicle_positions(date, missing_hours, trip_lookup)
    if not seg.empty:
        all_segments.append(seg)

new_segments = pd.concat(all_segments, ignore_index=True) if all_segments else pd.DataFrame()

# Combine cached + new
if not cached_segments.empty and not new_segments.empty:
    segments = pd.concat([cached_segments, new_segments], ignore_index=True)
elif not cached_segments.empty:
    segments = cached_segments
else:
    segments = new_segments

# Filter to bus vehicles only
segments = filter_bus_segments(segments)

# Deduplicate
dedup_cols = ["vehicle_id", "start_time", "end_time", "route_short_name"]
available = [c for c in dedup_cols if c in segments.columns]
segments = segments.drop_duplicates(subset=available, keep="last").reset_index(drop=True)

print(f"\nTotal segments (buses, {len(DATES)} days): {len(segments)}")
if not segments.empty:
    print(f"Unique vehicles: {segments['vehicle_id'].nunique()}")
    print(f"Routes observed: {segments['route_short_name'].nunique()}")
segments.head(10)

## 6. Detect Deadheads (Tomkörningar)

Identifies periods where buses travel empty between trips.

In [ ]:
import importlib
import analysis as _analysis_mod
importlib.reload(_analysis_mod)
from analysis import build_observed_deadheads, build_planned_deadheads, filter_deadheads_osrm, build_dwell_lookup

# Observed deadheads (from real-time vehicle tracking)
observed = build_observed_deadheads(segments, stops)
print(f"Observed deadheads (raw): {len(observed)}")

# Planned deadheads (from static GTFS schedule)
planned = build_planned_deadheads(trips, stop_times, stops, routes, operator_df)
print(f"Planned deadheads (raw): {len(planned)}")

# OSRM filter: remove deadheads with unrealistic durations
# (>100% slower or >50% faster than OSRM driving estimate)
if not observed.empty:
    observed = filter_deadheads_osrm(observed, min_ratio=0.5, max_ratio=2.0)
    print(f"Observed after OSRM filter: {len(observed)}")

# For planned: subtract observed dwell time per stop pair before OSRM comparison
# (planned duration includes idle time at terminals that observed data reveals)
if not planned.empty:
    dwell = build_dwell_lookup(observed)
    print(f"Dötidslookup: {len(dwell)} hållplatspar med observerad dötid")
    planned = filter_deadheads_osrm(planned, min_ratio=0.5, max_ratio=2.0, dwell_lookup=dwell)
    print(f"Planned after OSRM filter: {len(planned)}")

## 7. Save Results to CSV

Saves segments and deadheads to CSV files in the `data/` directory. Automatically deduplicates with any existing data.

In [ ]:
from csv_handler import save_segments, save_deadheads, push_data_to_git

save_segments(segments)

if not observed.empty:
    save_deadheads(observed)

if not planned.empty:
    save_deadheads(planned)

# OSRM cache is already saved by filter_deadheads_osrm, but ensure it's included in git push
print("\nResults saved to data/ directory.")

# Push data to git so next run can reuse it (segments, deadheads, and OSRM cache)
date_label_csv = f"{DATES[0]} to {DATES[-1]}" if len(DATES) > 1 else DATES[0]
push_data_to_git(f"Cache data for {date_label_csv}")

## 8. Analysis Summary

Overview statistics of the collected data.

In [ ]:
from utils import classify_period

date_label = f"{DATES[0]} to {DATES[-1]}" if len(DATES) > 1 else DATES[0]

print("=" * 60)
print(f"ANALYSIS SUMMARY FOR {date_label}")
print("=" * 60)

# Key stats
n_obs = len(observed) if not observed.empty else 0
n_unique = observed.drop_duplicates(subset=["from_stop_observed", "to_stop_observed"]).shape[0] if n_obs > 0 else 0
n_hours = len(HOURS) * len(DATES)

print(f"\nObserverade tomkörningar: {n_obs:,}")
print(f"Unika tomkörningar (hållplatspar): {n_unique:,}")
print(f"Analyserade timmar: {n_hours}")
print(f"Analysperiod: {DATES[0]} – {DATES[-1]}")

if n_obs > 0:
    print(f"\nSnitt restid: {observed['duration_min'].mean():.1f} min")
    if 'beräknad_körtid_min' in observed.columns:
        osrm_mean = observed['beräknad_körtid_min'].mean()
        print(f"Snitt beräknad tid (OSRM): {osrm_mean:.1f} min")
    print(f"\nPer dagtyp:")
    if 'day_type' in observed.columns:
        print(observed['day_type'].value_counts().to_string())
    print(f"\nPer trafikperiod:")
    print(observed['period'].value_counts().to_string())
    print(f"\nPer operatör:")
    print(observed['operator'].value_counts().to_string())

## 9. Visualizations

In [ ]:
import plotly.express as px

date_label = f"{DATES[0]} to {DATES[-1]}" if len(DATES) > 1 else DATES[0]

# Deadheads by operator
if not observed.empty:
    fig = px.histogram(
        observed, x="operator", color="period",
        title=f"Observed Deadheads by Operator & Traffic Period ({date_label})",
        labels={"operator": "Operator", "count": "Count", "period": "Traffic Period"},
        barmode="stack",
    )
    fig.update_layout(xaxis_categoryorder="total descending")
    fig.show()

In [ ]:
# Deadhead duration distribution
if not observed.empty:
    fig = px.histogram(
        observed, x="duration_min", nbins=30,
        title=f"Deadhead Duration Distribution ({date_label})",
        labels={"duration_min": "Duration (minutes)", "count": "Count"},
    )
    fig.show()

In [ ]:
# Top deadhead stop pairs
if not observed.empty:
    top_stops = (
        observed.groupby(["from_stop_observed", "to_stop_observed"])
        .agg(count=("vehicle_id", "size"), avg_duration=("duration_min", "mean"), avg_distance=("move_m", "mean"))
        .reset_index()
        .sort_values("count", ascending=False)
        .head(15)
    )
    top_stops["stop_pair"] = top_stops["from_stop_observed"] + " → " + top_stops["to_stop_observed"]
    top_stops["avg_duration"] = top_stops["avg_duration"].round(1)
    top_stops["avg_distance"] = (top_stops["avg_distance"] / 1000).round(1)

    fig = px.bar(
        top_stops, x="stop_pair", y="count",
        hover_data=["avg_duration", "avg_distance"],
        title=f"Top 15 Deadhead Stop Pairs ({date_label})",
        labels={"stop_pair": "Hållplatspar", "count": "Antal", "avg_duration": "Snitt min", "avg_distance": "Snitt km"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
# Deadhead timeline
if not observed.empty:
    timeline = observed.copy()
    timeline["hour"] = pd.to_datetime(timeline["deadhead_start"]).dt.hour
    hourly = timeline.groupby("hour").agg(
        count=("vehicle_id", "size"),
        total_km=("move_m", lambda x: (x.sum() / 1000).round(1)),
    ).reset_index()

    fig = px.bar(
        hourly, x="hour", y="count",
        hover_data=["total_km"],
        title=f"Deadheads by Hour of Day ({date_label})",
        labels={"hour": "Hour", "count": "Number of Deadheads", "total_km": "Total Distance (km)"},
    )
    fig.update_layout(xaxis_dtick=1)
    fig.show()

## 10. Fetch Delay Data & Build Line Stop Data

Fetches GTFS-RT TripUpdates for per-stop delay analysis and builds the line stop sequences for the map view.

In [ ]:
from fetcher import fetch_trip_updates, build_line_stop_data

# Fetch TripUpdates for delay data (optional — report works without it)
all_delays = []
for date in DATES:
    try:
        print(f"\n=== TripUpdates {date} ===")
        d = fetch_trip_updates(date, HOURS, trip_lookup)
        if not d.empty:
            all_delays.append(d)
            print(f"  {len(d):,} records")
    except Exception as e:
        print(f"  Could not fetch TripUpdates for {date}: {e}")

if all_delays:
    delays_df = pd.concat(all_delays, ignore_index=True)
    print(f"\nTotal delay data: {len(delays_df):,} records, {delays_df['route_short_name'].nunique()} routes")
else:
    delays_df = None
    print("\nNo delay data available.")

# Build line stop data for map view
line_stop_data = build_line_stop_data(routes, trips, stop_times, stops, delays_df)
print(f"Line stop data: {len(line_stop_data)} line-directions")

## 11. Generate HTML Report

Dark-themed HTML report with two tabs:
- **Tomkörningar**: Deadheads grouped by stop pair (not route) with period subtabs
- **Linjer**: Select a bus line to see it on the map with average delay per stop

In [ ]:
import importlib
import report as _report_mod
importlib.reload(_report_mod)
from report import generate_html_report

date_label = f"{DATES[0]}_to_{DATES[-1]}" if len(DATES) > 1 else DATES[0]
report_path = generate_html_report(
    observed, planned, segments, date_label,
    line_stop_data=line_stop_data,
    dates=DATES, hours=HOURS,
)

# Auto-download in Colab, or print path for local use
if IN_COLAB:
    from google.colab import files
    files.download(report_path)
    print("Download started!")
else:
    import webbrowser
    abs_path = os.path.abspath(report_path)
    print(f"Report saved to: {abs_path}")
    try:
        webbrowser.open(f"file://{abs_path}")
        print("Opened in browser.")
    except Exception:
        print("Open the file above in your browser to view the report.")

## 12. Explore Raw Data

Browse the raw dataframes interactively.

In [ ]:
# View observed deadheads
if not observed.empty:
    display_cols = ["operator", "prev_route", "next_route", "from_stop_observed", "to_stop_observed",
                    "deadhead_start", "duration_min", "beräknad_körtid_min", "move_m", "speed_kmh", "period", "day_type"]
    available = [c for c in display_cols if c in observed.columns]
    observed[available].head(20)
else:
    print("No observed deadheads to show.")